# Silver Transform: Sales Orders

**Layer:** Silver  
**Owner:** Data Engineering  
**Last Modified:** 2024-01-15  
**Schedule:** Daily at 04:00 UTC  

## Purpose
Reads raw sales order data from the Bronze lakehouse, applies:
- Schema enforcement and type casting
- Null handling and default values
- Currency normalisation to USD
- Deduplication
- SCD Type 2 merge into the Silver table
- Data quality checks via Great Expectations


In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
# These are injected by the pipeline at runtime.
# When running interactively, set defaults here.
processing_date = getArgument('processing_date', '2024-01-24')  # noqa: F821
run_id          = getArgument('run_id',          'INTERACTIVE')  # noqa: F821
full_refresh    = getArgument('full_refresh',    'false').lower() == 'true'  # noqa: F821

print(f'Processing date : {processing_date}')
print(f'Run ID          : {run_id}')
print(f'Full refresh    : {full_refresh}')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    DecimalType, IntegerType, BooleanType, TimestampType
)
from delta.tables import DeltaTable
from datetime import datetime, date
import hashlib

spark = SparkSession.builder.getOrCreate()
spark.conf.set('spark.sql.shuffle.partitions', '64')
spark.conf.set('spark.databricks.delta.optimizeWrite.enabled', 'true')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
BRONZE_TABLE  = 'contoso_analytics_lakehouse.bronze.sales_orders_raw'
SILVER_TABLE  = 'contoso_analytics_lakehouse.silver.sales_orders_cleansed'
FX_TABLE      = 'contoso_analytics_lakehouse.silver.fx_rates_daily'

MERGE_KEYS    = ['order_id']
HASH_COLUMNS  = ['customer_id', 'product_id', 'quantity', 'unit_price', 'status', 'region']

NOW_TS        = datetime.utcnow()
HIGH_DATE     = datetime(9999, 12, 31, 23, 59, 59)

In [ ]:
# ── Step 1: Read Bronze ────────────────────────────────────────────────────────
bronze_df = spark.table(BRONZE_TABLE)

if not full_refresh:
    # Incremental: only process rows from today's ingestion partition
    bronze_df = bronze_df.filter(F.col('ingestion_date') == processing_date)

print(f'Bronze rows read: {bronze_df.count():,}')

In [ ]:
# ── Step 2: Cast & cleanse ────────────────────────────────────────────────────
cleansed_df = (
    bronze_df
    .withColumn('order_date',   F.to_date('order_date', 'yyyy-MM-dd'))
    .withColumn('quantity',     F.col('quantity').cast(DecimalType(18, 4)))
    .withColumn('unit_price',   F.col('unit_price').cast(DecimalType(18, 4)))
    .withColumn('total_amount', F.col('total_amount').cast(DecimalType(18, 4)))
    # Standardise status codes from ERP
    .withColumn('status', F.when(F.col('status').isin(['C', 'COMP', 'COMPLETED']), 'Completed')
                           .when(F.col('status').isin(['P', 'PEND', 'PENDING']),   'Pending')
                           .when(F.col('status').isin(['X', 'CANC', 'CANCELLED']), 'Cancelled')
                           .otherwise('Unknown'))
    # Fill nulls
    .fillna({'region': 'UNKNOWN', 'sales_rep_id': 'UNASSIGNED'})
    # Partition helpers
    .withColumn('order_year',  F.year('order_date'))
    .withColumn('order_month', F.month('order_date'))
)

In [ ]:
# ── Step 3: Currency normalisation ────────────────────────────────────────────
fx_df = (
    spark.table(FX_TABLE)
    .filter(F.col('rate_date') == processing_date)
    .select('from_currency', 'rate_to_usd')
)

# Add USD row (rate = 1.0) so USD orders are also joined
from pyspark.sql import Row
usd_row = spark.createDataFrame([Row(from_currency='USD', rate_to_usd=1.0)])
fx_df = fx_df.union(usd_row)

cleansed_df = (
    cleansed_df
    .join(fx_df, cleansed_df['currency'] == fx_df['from_currency'], 'left')
    .withColumn('fx_rate_to_usd',   F.coalesce(F.col('rate_to_usd'), F.lit(1.0)))
    .withColumn('unit_price_usd',   F.round(F.col('unit_price')   * F.col('fx_rate_to_usd'), 4))
    .withColumn('total_amount_usd', F.round(F.col('total_amount') * F.col('fx_rate_to_usd'), 4))
    .withColumn('original_currency', F.col('currency'))
    .drop('from_currency', 'rate_to_usd', 'currency', 'unit_price', 'total_amount')
)

In [ ]:
# ── Step 4: Deduplication ─────────────────────────────────────────────────────
# Keep the latest ingestion of each order_id within the batch
from pyspark.sql.window import Window

w = Window.partitionBy('order_id').orderBy(F.desc('_ingestion_ts'))
cleansed_df = (
    cleansed_df
    .withColumn('_row_num', F.row_number().over(w))
    .filter(F.col('_row_num') == 1)
    .drop('_row_num')
)

print(f'Rows after dedup: {cleansed_df.count():,}')

In [ ]:
# ── Step 5: Add SCD2 & hash columns ───────────────────────────────────────────
def md5_hash(cols):
    return F.md5(F.concat_ws('|', *[F.coalesce(F.col(c).cast('string'), F.lit('')) for c in cols]))

staged_df = (
    cleansed_df
    .withColumn('_valid_from',   F.lit(NOW_TS).cast(TimestampType()))
    .withColumn('_valid_to',     F.lit(None).cast(TimestampType()))
    .withColumn('_is_current',   F.lit(True))
    .withColumn('_updated_ts',   F.lit(NOW_TS).cast(TimestampType()))
    .withColumn('_source_hash',  md5_hash(HASH_COLUMNS))
)

In [ ]:
# ── Step 6: SCD2 Merge into Silver ────────────────────────────────────────────
if full_refresh:
    staged_df.write.format('delta').mode('overwrite').saveAsTable(SILVER_TABLE)
    print('Full refresh complete.')
else:
    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

    # Expire old records where hash has changed
    silver_delta.alias('target').merge(
        staged_df.alias('source'),
        ' AND '.join([f'target.{k} = source.{k}' for k in MERGE_KEYS])
        + ' AND target._is_current = true'
    ).whenMatchedUpdate(
        condition='target._source_hash != source._source_hash',
        set={
            '_valid_to':   f'source._valid_from',
            '_is_current': 'false',
            '_updated_ts': f"'{NOW_TS}'"
        }
    ).execute()

    # Insert new / changed records
    (
        staged_df.join(
            spark.table(SILVER_TABLE).filter('_is_current = true').select(*MERGE_KEYS, '_source_hash'),
            on=MERGE_KEYS, how='left_anti'
        )
        .write.format('delta').mode('append').saveAsTable(SILVER_TABLE)
    )

    print('SCD2 merge complete.')

In [ ]:
# ── Step 7: Optimise Delta table ──────────────────────────────────────────────
spark.sql(f'OPTIMIZE {SILVER_TABLE} ZORDER BY (order_date, customer_id, product_id)')
print('OPTIMIZE complete.')

In [ ]:
# ── Step 8: Row count validation ──────────────────────────────────────────────
silver_count = spark.table(SILVER_TABLE).filter('_is_current = true').count()
print(f'Silver active rows (post-merge): {silver_count:,}')

# Fail if we have suspiciously few rows
assert silver_count > 0, 'ERROR: Silver table is empty after merge!'
print('All validations passed. Notebook complete.')